# GEMS TCO Missing Data Analysis — 2024 July

미싱 데이터 패턴 분석:
- 일별 / 시간별 미싱 개수 및 비율
- 공간적 분포 (어떤 격자가 자주 미싱인가)
- Day × Time heatmap (어느 날/시간대 조합이 나쁜가)
- 구름 등 원인 패턴 탐색

In [ ]:
import sys, pickle, re
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as mticker

sys.path.insert(0, "/Users/joonwonlee/Documents/GEMS_TCO-1/src")
from GEMS_TCO import configuration as config
from GEMS_TCO.data_loader import load_data_dynamic_processed

YEAR   = '2024'
MONTH  = 7
LAT_RANGE = [-3, 2]
LON_RANGE = [121, 131]

OUT_DIR = Path('/Users/joonwonlee/Documents/GEMS_TCO-1/Exercises/st_model/day/local_computer/log')
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('data path:', config.mac_data_load_path)

In [ ]:
# ── raw load (no ordering needed, is_whittle=True) ────────────────────
loader = load_data_dynamic_processed(config.mac_data_load_path)

coarse_dicts_raw, _, _, monthly_mean = loader.load_maxmin_ordered_data_bymonthyear(
    lat_lon_resolution=[1, 1],
    mm_cond_number=10,
    years_=[YEAR],
    months_=[MONTH],
    lat_range=LAT_RANGE,
    lon_range=LON_RANGE,
    is_whittle=True,    # skip expensive ordering
)

all_keys = sorted(coarse_dicts_raw.keys())
n_slots  = len(all_keys)
n_grid   = len(coarse_dicts_raw[all_keys[0]])

print(f'Monthly mean TCO : {monthly_mean:.4f} DU')
print(f'Time slots       : {n_slots}')
print(f'Grid cells       : {n_grid:,}')
print(f'Example key      : {all_keys[0]}')

In [ ]:
# ── parse metadata from keys ─────────────────────────────────────────
# key format after loader wrapping: '2024_07_y24m07day01_hm00:53'
# inner part: 'y24m07day01_hm00:53'

def parse_key(k):
    # strip year_month_ prefix added by loader
    inner = k.split('_', 2)[-1] if k.count('_') >= 2 else k
    m_day  = re.search(r'day(\d+)', inner)
    m_hour = re.search(r'hm(\d+):(\d+)', inner)
    day  = int(m_day.group(1))  if m_day  else -1
    hour = int(m_hour.group(1)) if m_hour else -1
    minute = int(m_hour.group(2)) if m_hour else 0
    return day, hour, minute

# verify on a few keys
for k in all_keys[:3] + all_keys[-2:]:
    d, h, mi = parse_key(k)
    print(f'  {k!r:50s} → day={d:2d}  hour={h:2d}:{mi:02d}')

In [ ]:
# ── per-slot missing stats ────────────────────────────────────────────
records = []
for key in all_keys:
    df  = coarse_dicts_raw[key]
    day, hour, minute = parse_key(key)
    o3 = pd.to_numeric(df['ColumnAmountO3'], errors='coerce')
    n_obs   = len(df)
    n_miss  = int(o3.isna().sum())
    pct_miss = n_miss / n_obs * 100
    records.append({
        'key':      key,
        'day':      day,
        'hour':     hour,
        'minute':   minute,
        'slot_in_day': hour * 60 + minute,
        'n_obs':    n_obs,
        'n_miss':   n_miss,
        'n_valid':  n_obs - n_miss,
        'pct_miss': pct_miss,
    })

df_slots = pd.DataFrame(records).sort_values(['day', 'hour', 'minute']).reset_index(drop=True)

print(f'Total observations : {df_slots.n_obs.sum():,}')
print(f'Total missing      : {df_slots.n_miss.sum():,}  ({df_slots.n_miss.sum()/df_slots.n_obs.sum()*100:.2f}%)')
print(f'Missing range      : {df_slots.pct_miss.min():.1f}% – {df_slots.pct_miss.max():.1f}%')
print(f'Median missing     : {df_slots.pct_miss.median():.1f}%')
print()
print(df_slots[['day','hour','n_obs','n_miss','pct_miss']].head(10).to_string(index=False))


In [ ]:
# ── 일별 집계 ─────────────────────────────────────────────────────────
df_daily = (
    df_slots.groupby('day')
    .agg(
        n_slots   = ('key',      'count'),
        n_obs     = ('n_obs',    'sum'),
        n_miss    = ('n_miss',   'sum'),
        pct_miss_mean = ('pct_miss', 'mean'),
        pct_miss_max  = ('pct_miss', 'max'),
        pct_miss_min  = ('pct_miss', 'min'),
    )
    .reset_index()
)
df_daily['pct_miss_total'] = df_daily['n_miss'] / df_daily['n_obs'] * 100

print('일별 미싱 요약:')
print(df_daily[['day','n_slots','n_obs','n_miss','pct_miss_total','pct_miss_max']].to_string(index=False))

In [ ]:
# ── 시간별 집계 (hour of day) ─────────────────────────────────────────
df_hourly = (
    df_slots.groupby('hour')
    .agg(
        n_slots        = ('key',      'count'),
        pct_miss_mean  = ('pct_miss', 'mean'),
        pct_miss_std   = ('pct_miss', 'std'),
        pct_miss_max   = ('pct_miss', 'max'),
    )
    .reset_index()
)
print('시간대별 미싱 요약 (월 전체 평균):')
print(df_hourly.to_string(index=False))

In [ ]:
# ── Figure 1: 일별 미싱 비율 + 시간별 평균 ───────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(16, 8))

# (a) daily
ax = axes[0]
ax.bar(df_daily['day'], df_daily['pct_miss_total'], color='steelblue', alpha=0.8, label='일별 총 미싱%')
daily_yerr = np.vstack([
    df_daily['pct_miss_mean'] - df_daily['pct_miss_min'],
    df_daily['pct_miss_max'] - df_daily['pct_miss_mean'],
])
ax.errorbar(
    df_daily['day'],
    df_daily['pct_miss_mean'],
    yerr=daily_yerr,
    fmt='o', color='tomato', ms=4, lw=1, capsize=3, label='슬롯별 평균 및 min-max 범위'
)
ax.axhline(5, color='grey', lw=1, linestyle='--', alpha=0.7, label='5% reference')
ax.set_xlabel('Day of July 2024')
ax.set_ylabel('Missing %')
ax.set_title('Daily Missing Rate — GEMS TCO 2024-07  (domain: lat[-3,2], lon[121,131])')
ax.set_xticks(df_daily['day'])
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# (b) hourly
ax2 = axes[1]
ax2.bar(df_hourly['hour'], df_hourly['pct_miss_mean'], color='darkorange', alpha=0.8)
ax2.errorbar(
    df_hourly['hour'],
    df_hourly['pct_miss_mean'],
    yerr=df_hourly['pct_miss_std'],
    fmt='k.', ms=3, lw=1, capsize=3
)
ax2.axhline(5, color='grey', lw=1, linestyle='--', alpha=0.7)
ax2.set_xlabel('Hour of Day (UTC)')
ax2.set_ylabel('Missing % (monthly mean ± std)')
ax2.set_title('Hourly Missing Rate — averaged over all July days')
ax2.set_xticks(df_hourly['hour'])
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUT_DIR / 'missing_daily_hourly_050626.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved.')


In [ ]:
# ── Figure 2: Day × Time slot heatmap ────────────────────────────────
# Pivot: rows=day, cols=time slot index within day
df_slots['slot_idx'] = df_slots.groupby('day').cumcount()
pivot = df_slots.pivot(index='day', columns='slot_idx', values='pct_miss')

# Tick labels: hour:minute for each slot (use median times across days)
slot_times = (
    df_slots.groupby('slot_idx')[['hour','minute']]
    .agg(lambda x: x.mode()[0])
    .reset_index()
)
slot_labels = [f"{int(r.hour):02d}:{int(r.minute):02d}" for _, r in slot_times.iterrows()]

fig, ax = plt.subplots(figsize=(18, 9))
im = ax.imshow(
    pivot.values,
    aspect='auto',
    cmap='YlOrRd',
    vmin=0, vmax=min(pivot.values.max(), 50),
    interpolation='nearest',
)
plt.colorbar(im, ax=ax, label='Missing %')
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels([f'Day {d}' for d in pivot.index], fontsize=8)
ax.set_xticks(range(len(slot_labels)))
ax.set_xticklabels(slot_labels, fontsize=8, rotation=45, ha='right')
ax.set_xlabel('Time slot (UTC)')
ax.set_ylabel('Day')
ax.set_title('Missing % — Day × Time Slot  |  GEMS TCO 2024-07  lat[-3,2] lon[121,131]')

# Annotate cells above 20%
for r in range(pivot.shape[0]):
    for c in range(pivot.shape[1]):
        v = pivot.values[r, c]
        if not np.isnan(v) and v > 20:
            ax.text(c, r, f'{v:.0f}', ha='center', va='center',
                    fontsize=6, color='black', fontweight='bold')

plt.tight_layout()
plt.savefig(OUT_DIR / 'missing_heatmap_day_slot_050626.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved.')

In [ ]:
# ── 공간 분석: 격자별 미싱 빈도 ───────────────────────────────────────
# 모든 슬롯에 걸쳐 각 격자 셀이 몇 번 미싱인지 카운트

# 첫 번째 슬롯에서 격자 좌표 고정
ref_df   = coarse_dicts_raw[all_keys[0]].reset_index(drop=True)
n_cells  = len(ref_df)
lats     = ref_df['Latitude'].values
lons     = ref_df['Longitude'].values

miss_count  = np.zeros(n_cells, dtype=np.int32)
total_count = np.zeros(n_cells, dtype=np.int32)

for key in all_keys:
    df = coarse_dicts_raw[key].reset_index(drop=True)
    if len(df) != n_cells:
        print(f'Warning: {key} has {len(df)} rows, expected {n_cells}')
        continue
    is_miss = df['ColumnAmountO3'].isna().values
    miss_count  += is_miss.astype(np.int32)
    total_count += 1

miss_frac = miss_count / total_count  # fraction of slots where each cell is missing

print(f'Grid cells with 0% missing  : {(miss_frac == 0).sum():,}')
print(f'Grid cells with >10% missing: {(miss_frac > 0.10).sum():,}')
print(f'Grid cells with >50% missing: {(miss_frac > 0.50).sum():,}')
print(f'Grid cells with 100% missing: {(miss_frac == 1.0).sum():,}')
print(f'Max missing fraction: {miss_frac.max():.3f}')
print(f'Mean missing fraction: {miss_frac.mean():.3f}')

In [ ]:
# ── Figure 3: 공간 분포 — 격자별 미싱 비율 ───────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# (a) scatter by missing fraction
ax = axes[0]
sc = ax.scatter(
    lons, lats,
    c=miss_frac * 100,
    cmap='YlOrRd',
    vmin=0, vmax=100,
    s=4, alpha=0.8, linewidths=0
)
plt.colorbar(sc, ax=ax, label='Missing % (over all July slots)')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Spatial distribution of missing rate\n2024-07, all time slots')
ax.set_aspect('equal')
ax.grid(alpha=0.3)

# (b) histogram of missing fractions
ax2 = axes[1]
bins = np.linspace(0, 1, 51)
ax2.hist(miss_frac, bins=bins, color='steelblue', edgecolor='white', linewidth=0.3)
ax2.set_xlabel('Missing fraction per grid cell')
ax2.set_ylabel('Number of grid cells')
ax2.set_title('Histogram of per-cell missing fractions')
ax2.axvline(miss_frac.mean(), color='tomato', lw=2, linestyle='--', label=f'mean={miss_frac.mean():.3f}')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUT_DIR / 'missing_spatial_050626.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved.')

In [ ]:
# ── Figure 4: 위도/경도 별 미싱 패턴 ─────────────────────────────────
unique_lats = np.sort(np.unique(lats))[::-1]   # north to south
unique_lons = np.sort(np.unique(lons))          # west to east
n_lat = len(unique_lats)
n_lon = len(unique_lons)

lat_to_row = {la: i for i, la in enumerate(unique_lats)}
lon_to_col = {lo: i for i, lo in enumerate(unique_lons)}

miss_grid = np.full((n_lat, n_lon), np.nan)
for i, (la, lo) in enumerate(zip(lats, lons)):
    r = lat_to_row.get(la)
    c = lon_to_col.get(lo)
    if r is not None and c is not None:
        miss_grid[r, c] = miss_frac[i] * 100

mapped_cells = np.isfinite(miss_grid).sum()
print(f'Mapped grid cells: {mapped_cells:,} / {n_cells:,}')
if mapped_cells != n_cells:
    print('Warning: not all coordinate cells were mapped into miss_grid.')

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# (a) full grid imshow
ax = axes[0]
im = ax.imshow(
    miss_grid,
    aspect='auto',
    cmap='YlOrRd',
    vmin=0, vmax=100,
    extent=[unique_lons[0], unique_lons[-1], unique_lats[-1], unique_lats[0]],
)
plt.colorbar(im, ax=ax, label='Missing %')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Missing % per grid cell\n(July 2024, all slots)')

# (b) aggregated by latitude band
ax2 = axes[1]
lat_miss = np.nanmean(miss_grid, axis=1)  # mean over lon for each lat row
ax2.barh(unique_lats, lat_miss, height=0.035, color='steelblue', alpha=0.7)
ax2.set_xlabel('Mean missing % (averaged over longitude)')
ax2.set_ylabel('Latitude')
ax2.set_title('Missing % by latitude band')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUT_DIR / 'missing_grid_latband_050626.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Grid shape: {n_lat} lat × {n_lon} lon')


In [ ]:
# ── Figure 5: 일별 공간 패턴 (선택한 며칠) ────────────────────────────
# 하루 8 슬롯을 합쳐서 그 날의 격자별 미싱 비율 계산

SELECTED_DAYS = [1, 5, 10, 15, 20, 25, 31]   # 필요시 수정
SELECTED_DAYS = [d for d in SELECTED_DAYS if d in df_daily['day'].values]

fig, axes = plt.subplots(2, 4, figsize=(22, 10))
axes = axes.flatten()

def day_miss_grid(target_day):
    day_keys = [k for k in all_keys if parse_key(k)[0] == target_day]
    mc = np.zeros(n_cells, dtype=np.int32)
    tc = np.zeros(n_cells, dtype=np.int32)
    for key in day_keys:
        df = coarse_dicts_raw[key].reset_index(drop=True)
        if len(df) != n_cells:
            continue
        o3 = pd.to_numeric(df['ColumnAmountO3'], errors='coerce')
        mc += o3.isna().values.astype(np.int32)
        tc += 1
    frac = np.where(tc > 0, mc / tc, np.nan)
    grid = np.full((n_lat, n_lon), np.nan)
    for i, (la, lo) in enumerate(zip(lats, lons)):
        r = lat_to_row.get(la)
        c = lon_to_col.get(lo)
        if r is not None and c is not None:
            grid[r, c] = frac[i] * 100
    return grid, len(day_keys)

for i, day in enumerate(SELECTED_DAYS[:7]):
    grid, n_day_slots = day_miss_grid(day)
    row_info = df_daily[df_daily.day == day].iloc[0]
    im = axes[i].imshow(
        grid, aspect='auto', cmap='YlOrRd', vmin=0, vmax=100,
        extent=[unique_lons[0], unique_lons[-1], unique_lats[-1], unique_lats[0]],
    )
    plt.colorbar(im, ax=axes[i])
    axes[i].set_title(
        f'Day {day}  ({n_day_slots} slots)\n'
        f'miss={row_info.pct_miss_total:.1f}%  max_slot={row_info.pct_miss_max:.0f}%',
        fontsize=9
    )
    axes[i].set_xlabel('Lon', fontsize=7)
    axes[i].set_ylabel('Lat', fontsize=7)

# last panel: full month average
im = axes[7].imshow(
    miss_grid, aspect='auto', cmap='YlOrRd', vmin=0, vmax=100,
    extent=[unique_lons[0], unique_lons[-1], unique_lats[-1], unique_lats[0]],
)
plt.colorbar(im, ax=axes[7])
axes[7].set_title('ALL JULY (mean)', fontsize=9)
axes[7].set_xlabel('Lon', fontsize=7)
axes[7].set_ylabel('Lat', fontsize=7)

fig.suptitle('Daily spatial missing patterns — GEMS TCO 2024-07\nlat[-3,2] lon[121,131]', fontsize=12)
plt.tight_layout()
plt.savefig(OUT_DIR / 'missing_daily_spatial_050626.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved.')


In [ ]:
# ── Figure 6: 연속 미싱 패턴 — 같은 슬롯 위치가 계속 비는지 ─────────
# slot_idx 별로 전 월 통합 미싱률 계산

slot_ids = sorted(df_slots['slot_idx'].dropna().unique().astype(int))
slot_grids = []
for si in slot_ids:
    slot_keys = df_slots[df_slots.slot_idx == si]['key'].values
    mc = np.zeros(n_cells, dtype=np.int32)
    tc = np.zeros(n_cells, dtype=np.int32)
    for key in slot_keys:
        if key not in coarse_dicts_raw:
            continue
        df = coarse_dicts_raw[key].reset_index(drop=True)
        if len(df) != n_cells:
            continue
        o3 = pd.to_numeric(df['ColumnAmountO3'], errors='coerce')
        mc += o3.isna().values.astype(np.int32)
        tc += 1
    frac = np.where(tc > 0, mc / tc, np.nan)
    grid = np.full((n_lat, n_lon), np.nan)
    for i, (la, lo) in enumerate(zip(lats, lons)):
        r = lat_to_row.get(la)
        c = lon_to_col.get(lo)
        if r is not None and c is not None:
            grid[r, c] = frac[i] * 100
    slot_grids.append(grid)

# use median time for slot label
slot_time_labels = [
    f"{int(df_slots[df_slots.slot_idx==si]['hour'].median()):02d}:"
    f"{int(df_slots[df_slots.slot_idx==si]['minute'].median()):02d}"
    for si in slot_ids
]

n_cols = min(4, len(slot_ids))
n_rows = int(np.ceil(len(slot_ids) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5.5 * n_cols, 4.5 * n_rows), squeeze=False)
axes = axes.flatten()
for ax in axes[len(slot_ids):]:
    ax.axis('off')
for ax, si, grid, label in zip(axes, slot_ids, slot_grids, slot_time_labels):
    im = ax.imshow(
        grid, aspect='auto', cmap='YlOrRd', vmin=0, vmax=100,
        extent=[unique_lons[0], unique_lons[-1], unique_lats[-1], unique_lats[0]],
    )
    plt.colorbar(im, ax=ax)
    ax.set_title(f'Slot {si} (~{label} UTC)', fontsize=9)
    ax.set_xlabel('Lon', fontsize=7)
    ax.set_ylabel('Lat', fontsize=7)

fig.suptitle('Per-time-slot missing % (averaged over all July days)\nlat[-3,2] lon[121,131]', fontsize=12)
plt.tight_layout()
plt.savefig(OUT_DIR / 'missing_per_slot_spatial_050626.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved.')


In [ ]:
# ── Figure 7: 일별 미싱 time-series + 이동평균 ────────────────────────
fig, ax = plt.subplots(figsize=(18, 5))

# 슬롯별 시계열 (raw)
df_slots_sorted = df_slots.sort_values(['day', 'slot_idx'])
x = np.arange(len(df_slots_sorted))
ax.plot(x, df_slots_sorted['pct_miss'].values, color='steelblue', lw=0.8, alpha=0.6, label='per slot')

# 하루 슬롯 수 기준 이동평균
slots_per_day = df_slots.groupby('day').size()
rolling_window = int(slots_per_day.median())
ma = pd.Series(df_slots_sorted['pct_miss'].values).rolling(rolling_window, center=True).mean()
ax.plot(x, ma.values, color='tomato', lw=2, label=f'{rolling_window}-slot rolling mean (≈daily)')

# Day 구분선
cum = 0
for day, n_s in slots_per_day.items():
    ax.axvline(cum, color='grey', lw=0.5, alpha=0.5)
    ax.text(cum + n_s / 2, ax.get_ylim()[1] * 0.95, str(day), ha='center', fontsize=7, color='grey')
    cum += n_s

ax.axhline(5, color='grey', lw=1, linestyle='--', alpha=0.7, label='5% reference')
ax.set_xlabel('Time slot index')
ax.set_ylabel('Missing %')
ax.set_title('Time series of missing % per slot — GEMS TCO 2024-07')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / 'missing_timeseries_050626.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved.')


In [ ]:
# ── 요약 통계 테이블 ──────────────────────────────────────────────────
print('=' * 60)
print('MISSING DATA SUMMARY — GEMS TCO 2024-07')
print(f'Domain: lat {LAT_RANGE}, lon {LON_RANGE}')
print('=' * 60)
print(f'Grid cells per slot : {n_grid:,}')
print(f'Time slots total    : {n_slots}')
print(f'Slots per day       : {n_slots / 31:.1f} (median={df_slots.groupby("day").size().median():.0f})')
print()
print('Overall missing:')
print(f'  Total obs          : {df_slots.n_obs.sum():,}')
print(f'  Total missing      : {df_slots.n_miss.sum():,} ({df_slots.n_miss.sum()/df_slots.n_obs.sum()*100:.2f}%)')
print()
print('Per-slot missing %:')
for q in [0, 10, 25, 50, 75, 90, 100]:
    print(f'  p{q:3d}: {np.percentile(df_slots.pct_miss, q):.2f}%')
print()
print('Worst 5 slots:')
print(df_slots.nlargest(5, 'pct_miss')[['key','day','hour','n_miss','pct_miss']].to_string(index=False))
print()
print('Best 5 slots (lowest missing):')
print(df_slots.nsmallest(5, 'pct_miss')[['key','day','hour','n_miss','pct_miss']].to_string(index=False))
print()
print('Worst 5 days (총 미싱%): ')
print(df_daily.nlargest(5, 'pct_miss_total')[['day','n_miss','pct_miss_total','pct_miss_max']].to_string(index=False))
print()
print('Spatial: cells always missing (100%):', (miss_frac == 1).sum())
print('Spatial: cells never missing  (0%)  :', (miss_frac == 0).sum())

# Save summary
df_slots.to_csv(OUT_DIR / 'missing_slot_stats_050626.csv', index=False)
df_daily.to_csv(OUT_DIR / 'missing_daily_stats_050626.csv', index=False)
print('\nCSV saved.')